# LCG.py quantile log block 크래시 검증

**가설**: `LCG.py:194` (현재는 주석)의 `torch.quantile(src_str_scaled.flatten(), ...)` 호출이
`B * M * N^2` numel이 `2^24 = 16,777,216`을 넘으면 `RuntimeError: quantile() input tensor is too large` 로 터진다.

**검증 방식**:
- LCG forward의 핵심 텐서 expansion (line 306-309)과 scale 로직 (line 163-171) 만 떼서 재현
- 두 버전 비교: (A) quantile log 블록 살아있을 때, (B) 주석처리된 현재 상태
- 다양한 N으로 임계 확인: heart-like(N=13), hirid 추정(N=50), ICU 경계(N=100/120/200)


In [1]:
import torch
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device         : {device}")
print(f"torch version  : {torch.__version__}")
print(f"quantile limit : 2^24 = {2**24:,} elements")

Device         : cpu
torch version  : 2.0.1+cu117
quantile limit : 2^24 = 16,777,216 elements


In [2]:
def _build_tensors(B, M, N, D, K):
    """LCG.forward의 expansion (line 300-309) + scale (line 163-171) 재현."""
    src_feat = torch.randn(B, N, D, device=device, requires_grad=True)
    tgt_feat_lcg = torch.randn(M, K, D, device=device)
    src_str = torch.randn(B, N, N, device=device)
    tgt_str_lcg = torch.randn(M, K, K, device=device)

    # B * M expansion
    src_feat_exp = src_feat.unsqueeze(1).expand(B, M, N, D).reshape(B * M, N, D)
    src_str_exp  = src_str.unsqueeze(1).expand(B, M, N, N).reshape(B * M, N, N)
    lcg_feat_exp = tgt_feat_lcg.unsqueeze(0).expand(B, M, K, D).reshape(B * M, K, D)
    lcg_str_exp  = tgt_str_lcg.unsqueeze(0).expand(B, M, K, K).reshape(B * M, K, K)

    # cosine cost (default)
    src_norm = F.normalize(src_feat_exp, dim=-1)
    tgt_norm = F.normalize(lcg_feat_exp, dim=-1)
    cos_sim = torch.bmm(src_norm, tgt_norm.transpose(1, 2))
    M_cost = 1.0 - torch.exp(-(1.0 - cos_sim))

    # scale_ratio
    with torch.no_grad():
        feat_mean = M_cost.detach().mean()
        struct_mean_src = src_str_exp.detach().mean()
        struct_mean_tgt = lcg_str_exp.detach().mean()
        struct_mean = (struct_mean_src + struct_mean_tgt) / 2
        scale_ratio = feat_mean / struct_mean.clamp_min(1e-8)

    src_str_scaled = src_str_exp * scale_ratio
    tgt_str_scaled = lcg_str_exp * scale_ratio
    return src_feat, src_str_scaled, tgt_str_scaled, M_cost

In [3]:
def fgw_with_log_block(B, M, N, D, K):
    """LCG.py:174-204 의 quantile log 블록이 살아있는 버전."""
    src_feat, src_str_scaled, tgt_str_scaled, M_cost = _build_tensors(B, M, N, D, K)

    # ===== 원본 debug log block (line 174-204) 재현 =====
    log_step, log_interval = 10, 10  # 항상 트리거되도록
    if log_step % log_interval == 0 and src_feat.requires_grad:
        with torch.no_grad():
            y = M_cost.detach().float().flatten()
            qy = torch.quantile(y, torch.tensor([0.0, 0.5, 0.9, 0.95, 0.99, 1.0], device=y.device))

            cs = src_str_scaled.detach().float().flatten()
            ct = tgt_str_scaled.detach().float().flatten()
            qcs = torch.quantile(cs, torch.tensor([0.0, 0.5, 0.9, 0.95, 0.99, 1.0], device=cs.device))
            qct = torch.quantile(ct, torch.tensor([0.0, 0.5, 0.9, 0.95, 0.99, 1.0], device=ct.device))
    return cs.numel()


def fgw_without_log_block(B, M, N, D, K):
    """현재 LCG.py 상태 (172-204 모두 주석처리됨)."""
    src_feat, src_str_scaled, tgt_str_scaled, M_cost = _build_tensors(B, M, N, D, K)
    # quantile 호출 없음 → solve_gromov_batch로 바로 진입하는 것과 동일
    return src_str_scaled.numel()

In [4]:
def run(name, B, M, N, D=64, K=12):
    numel = B * M * N * N
    over = "OVER" if numel > 2**24 else "ok"

    # with-log = 원본 LCG.py:174-204 quantile log block 살아있는 버전 (옛날 코드)
    try:
        fgw_with_log_block(B, M, N, D, K)
        with_status = "OK"
        with_err = ""
    except RuntimeError as e:
        with_status = "CRASH"
        with_err = str(e).splitlines()[0][:60]
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    # without-log = 현재 LCG.py 상태 (172-204 모두 주석)
    try:
        fgw_without_log_block(B, M, N, D, K)
        without_status = "OK"
        without_err = ""
    except RuntimeError as e:
        without_status = "CRASH"
        without_err = str(e).splitlines()[0][:60]
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    print(f"{name:<22} | B={B:>3} M={M:>2} N={N:>3} | numel(B*M*N^2)={numel:>13,} [{over:>4}] | with-log(원본): {with_status:<5} | without-log(현재): {without_status:<5}")
    if with_err:
        print(f"     >> with-log error: {with_err}")
    return with_status, without_status


# ICU mortality 6개 dataset의 실제 feature 수 (meta.json)
configs = [
    ("support_mortality",  128, 12,  79),
    ("eicu_mortality",     128, 12, 144),
    ("mimic_mortality",    128, 12, 236),
    ("zigong_mortality",   128, 12, 236),
    ("sic_mortality",      128, 12, 246),
    ("hirid_mortality",    128, 12, 283),
]

print(f"임계 N (B=128, M=12): sqrt(2^24 / 1536) ≈ {(2**24 / (128*12))**0.5:.1f}\n")
print("="*150)
results = []
for cfg in configs:
    results.append((cfg, run(*cfg)))
print("="*150)

임계 N (B=128, M=12): sqrt(2^24 / 1536) ≈ 104.5

support_mortality      | B=128 M=12 N= 79 | numel(B*M*N^2)=    9,586,176 [  ok] | with-log(원본): OK    | without-log(현재): OK   
eicu_mortality         | B=128 M=12 N=144 | numel(B*M*N^2)=   31,850,496 [OVER] | with-log(원본): CRASH | without-log(현재): OK   
     >> with-log error: quantile() input tensor is too large
mimic_mortality        | B=128 M=12 N=236 | numel(B*M*N^2)=   85,549,056 [OVER] | with-log(원본): CRASH | without-log(현재): OK   
     >> with-log error: quantile() input tensor is too large
zigong_mortality       | B=128 M=12 N=236 | numel(B*M*N^2)=   85,549,056 [OVER] | with-log(원본): CRASH | without-log(현재): OK   
     >> with-log error: quantile() input tensor is too large
sic_mortality          | B=128 M=12 N=246 | numel(B*M*N^2)=   92,952,576 [OVER] | with-log(원본): CRASH | without-log(현재): OK   
     >> with-log error: quantile() input tensor is too large
hirid_mortality        | B=128 M=12 N=283 | numel(B*M*N^2)=  123,016,704 [

## 예상 결과 (실제 ICU mortality 6 dataset feature 수)

| dataset | N (features) | B*M*N² | 한도 16.78M | with-log(원본) | without-log(현재) |
|---|---|---|---|---|---|
| support_mortality | 79 | 9.59M | under | OK | OK |
| eicu_mortality | 144 | 31.85M | OVER | **CRASH** | OK |
| mimic_mortality | 236 | 85.55M | OVER | **CRASH** | OK |
| zigong_mortality | 236 | 85.55M | OVER | **CRASH** | OK |
| sic_mortality | 246 | 92.97M | OVER | **CRASH** | OK |
| hirid_mortality | 283 | 123.02M | OVER | **CRASH** | OK |

**해석**:
- 6개 source 중 5개가 임계 N≈104를 넘음 → multi-source 학습에서 어떤 batch가 들어와도 거의 100% with-log 버전은 터짐
- support 만 underflow → 만약 sweep을 support 단독으로 돌렸다면 with-log 버전도 안 터졌을 것
- 현재 LCG.py(without-log)는 hirid N=283에서도 정상 → 크래시 완전 해결
- 임계 N ≈ sqrt(16777216 / 1536) ≈ 104.5

In [5]:
# 추가 검증: solve_gromov_batch 자체는 N=200 even에서도 잘 도는지 빠르게 확인
import sys
sys.path.insert(0, '/home/eungyeop/LLM/tabular/ProtoLLM_entropic20251217')
try:
    from ot.batch import solve_gromov_batch
    B, M, N, D, K = 128, 12, 200, 64, 12
    _, src_str_scaled, tgt_str_scaled, M_cost = _build_tensors(B, M, N, D, K)
    a = torch.ones(B*M, N, device=device) / N
    b = torch.ones(B*M, K, device=device) / K
    print(f"solve_gromov_batch with B*M={B*M}, N={N}, K={K} ...")
    result = solve_gromov_batch(
        src_str_scaled, tgt_str_scaled, M=M_cost, alpha=1.0, reg=0.1, a=a, b=b,
        max_iter=2, max_iter_inner=5, tol=1e-3, grad='envelope'
    )
    print(f"  result.value shape : {result.value.shape}")
    print(f"  result.plan shape  : {result.plan.shape}")
    print("  → solve_gromov_batch는 quantile 한도와 무관, 정상 작동")
except ImportError as e:
    print(f"(solve_gromov_batch import 실패 - skip): {e}")
except Exception as e:
    print(f"solve_gromov_batch error: {e}")

solve_gromov_batch with B*M=1536, N=200, K=12 ...
  result.value shape : torch.Size([1536])
  result.plan shape  : torch.Size([1536, 200, 12])
  → solve_gromov_batch는 quantile 한도와 무관, 정상 작동
